# Supervised term-count features + lag testing + model comparison (standalone)





In [1]:
!pip install -q scikit-learn xgboost statsmodels pandas tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 MB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 80.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.9/118.9 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.3/233.3 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.4/252.4 MB 1.4 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = "/content/drive/MyDrive/brexit_covid_datasets"
print("Reading from:", DATA_DIR)

Mounted at /content/drive
Reading from: /content/drive/MyDrive/brexit_covid_datasets


In [3]:
import pandas as pd
import numpy as np
from datetime import datetime

ECON_INDICATOR_PATH = f"{DATA_DIR}/economic_indicator.csv"
OUTLETS = ["bbc", "dailymail", "guardian"]
TOPICS = ["brexit", "covid", "energy_crisis"]
AGG_FREQ = "MS"
TEST_FRACTION = 0.2
RANDOM_STATE = 42

LAGS_MONTHS = [0, 1, 2, 3, 6]   # 0 = same-month (nowcasting-style, kept as a baseline
                                 # for comparison); 1-6 = genuine forward prediction

RECESSION_WINDOWS = [
    {"name": "2020 COVID recession", "start": pd.Timestamp("2020-01-01"), "end": pd.Timestamp("2020-06-30")},
    {"name": "2023 H2 recession",    "start": pd.Timestamp("2023-07-01"), "end": pd.Timestamp("2023-12-31")},
]
LEADUP_MONTHS = 6
CALM_COMPARISON_MONTHS = 6

TFIDF_CANDIDATE_FEATURES = 300   # wide candidate vocabulary, ranked by raw frequency —
                                   # this is NOT the final feature count, just the pool
                                   # SelectKBest chooses from below
N_SELECTED_FEATURES = 30          # final feature count after selection. Cutting straight to
                                   # 30 by frequency (last version) fixed the overfitting but
                                   # kept whichever words were most COMMON, not most PREDICTIVE
                                   # of GDP growth. SelectKBest instead scores all 300
                                   # candidates by their actual relationship to growth (on
                                   # training data only) and keeps the best 30 of those.
TFIDF_MIN_DF = 2           # a term must appear in at least 2 different months
MAX_CHARS_PER_ARTICLE = 2000   # cap per-article length before concatenating, same
                                 # truncation used for FinBERT elsewhere in this project

## Load articles per (outlet, topic), then combine into one monthly document per outlet

In [4]:
def load_outlet_topic(outlet, topic):
    if topic == "energy_crisis":
        path = f"{DATA_DIR}/{outlet}_energy_crisis_TEST.csv"
    else:
        path = f"{DATA_DIR}/{outlet}_brexit_covid_text_FINAL.csv"
    df = pd.read_csv(path)
    df = df[df["topic"] == topic].copy()
    df["publish_date"] = pd.to_datetime(df["publish_date"], errors="coerce", utc=True).dt.tz_localize(None)
    df = df.dropna(subset=["publish_date", "text"]).sort_values("publish_date").reset_index(drop=True)
    return df

datasets = {}
for outlet in OUTLETS:
    for topic in TOPICS:
        key = (outlet, topic)
        try:
            df = load_outlet_topic(outlet, topic)
            datasets[key] = df
            print(f"{outlet}/{topic}: {len(df)} articles, {df['publish_date'].min()} to {df['publish_date'].max()}")
        except FileNotFoundError:
            print(f"{outlet}/{topic}: FILE NOT FOUND")

bbc/brexit: 794 articles, 2012-06-17 00:00:00 to 2026-07-21 00:00:00
bbc/covid: 790 articles, 2020-01-10 00:00:00 to 2026-08-08 00:00:00
bbc/energy_crisis: 800 articles, 2021-08-06 00:00:00 to 2026-08-12 00:00:00
dailymail/brexit: 799 articles, 2012-01-28 00:00:00 to 2026-03-16 00:00:00
dailymail/covid: 798 articles, 2020-01-05 00:00:00 to 2026-03-16 00:00:00
dailymail/energy_crisis: 799 articles, 2021-08-07 00:00:00 to 2026-03-16 00:00:00
guardian/brexit: 800 articles, 2012-02-10 21:00:04 to 2026-08-04 04:00:12
guardian/covid: 800 articles, 2020-03-23 20:41:20 to 2026-08-08 20:03:34
guardian/energy_crisis: 800 articles, 2021-08-03 05:31:14 to 2026-08-13 14:54:03


In [5]:
def build_monthly_corpus(outlets):
    """Concatenates every article (all 3 topics, and every outlet passed in) published
    in a given month into one big text blob for that month -> one row per month.
    Pass a single outlet in a list for the per-outlet corpora (matches your existing
    pipeline design); pass all three outlets together to build the POOLED corpus —
    testing whether pooling across sources, the way Kalamara et al. (2022) and the
    other literature sources do (blending outlets specifically to avoid outlet-specific
    bias, rather than modeling each one separately), behaves differently from keeping
    BBC/Daily Mail/Guardian as three separate models."""
    frames = [datasets[(o, t)] for o in outlets for t in TOPICS if (o, t) in datasets and len(datasets[(o, t)])]
    if not frames:
        return pd.Series(dtype=object)
    all_articles = pd.concat(frames, ignore_index=True)
    all_articles["text"] = all_articles["text"].astype(str).str.slice(0, MAX_CHARS_PER_ARTICLE)
    all_articles["month"] = all_articles["publish_date"].dt.to_period("M").dt.to_timestamp()
    monthly_docs = all_articles.groupby("month")["text"].apply(lambda texts: " ".join(texts))
    monthly_docs.index = pd.DatetimeIndex(monthly_docs.index)
    return monthly_docs.sort_index()

monthly_corpora = {outlet: build_monthly_corpus([outlet]) for outlet in OUTLETS}
monthly_corpora["pooled"] = build_monthly_corpus(OUTLETS)   # BBC + Daily Mail + Guardian, combined

OUTLETS_PLUS_POOLED = OUTLETS + ["pooled"]

for outlet, corpus in monthly_corpora.items():
    print(f"{outlet}: {len(corpus)} months of combined text, {corpus.index.min()} to {corpus.index.max()}")

bbc: 112 months of combined text, 2012-06-01 00:00:00 to 2026-08-01 00:00:00
dailymail: 113 months of combined text, 2012-01-01 00:00:00 to 2026-03-01 00:00:00
guardian: 137 months of combined text, 2012-02-01 00:00:00 to 2026-08-01 00:00:00
pooled: 165 months of combined text, 2012-01-01 00:00:00 to 2026-08-01 00:00:00


## GDP growth target

In [6]:
econ = pd.read_csv(ECON_INDICATOR_PATH)
econ["date"] = pd.to_datetime(econ["date"])
econ = econ.set_index("date").resample(AGG_FREQ).mean().rename(columns={"value": "econ_value"})
econ["growth"] = econ["econ_value"].pct_change() * 100
print(f"GDP growth series: {econ.index.min()} to {econ.index.max()}, {len(econ)} months")

GDP growth series: 1997-01-01 00:00:00 to 2026-05-01 00:00:00, 353 months


## UK Economic Policy Uncertainty (EPU) Index — Baker, Bloom & Davis

Pulled live from policyuncertainty

In [8]:
import requests
from io import BytesIO
!pip install openpyxl -q

EPU_URL = "https://www.policyuncertainty.com/media/UK_Policy_Uncertainty_Data.xlsx"
resp = requests.get(EPU_URL, timeout=20)
epu_raw = pd.read_excel(BytesIO(resp.content))
epu_raw.columns = [str(c).strip() for c in epu_raw.columns]
print("EPU file columns:", list(epu_raw.columns))

year_col = next(c for c in epu_raw.columns if c.lower() == "year")
month_col = next(c for c in epu_raw.columns if c.lower() == "month")
value_col = next(c for c in epu_raw.columns if c not in (year_col, month_col))

epu = epu_raw[[year_col, month_col, value_col]].dropna()
epu["date"] = pd.to_datetime(dict(year=epu[year_col].astype(int), month=epu[month_col].astype(int), day=1))
epu = epu.set_index("date")[value_col].rename("epu").sort_index()
epu = epu[~epu.index.duplicated(keep="last")]
print(f"UK EPU index: {epu.index.min()} to {epu.index.max()}, {len(epu)} months")
print(epu.tail())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.9/250.9 kB 9.6 MB/s eta 0:00:00
EPU file columns: ['year', 'month', 'UK_EPU_Index']
UK EPU index: 1998-01-01 00:00:00 to 2026-07-01 00:00:00, 343 months
date
2026-03-01    310.822658
2026-04-01    309.342030
2026-05-01    301.639599
2026-06-01    207.985950
2026-07-01    243.872883
Name: epu, dtype: float64


## Models — Ridge, Random Forest, Gradient Boosting, naive + ARIMA baselines

In [9]:
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from statsmodels.tsa.arima.model import ARIMA

try:
    from xgboost import XGBRegressor
    def make_gbm():
        return XGBRegressor(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=RANDOM_STATE, verbosity=0)
    GBM_NAME = "XGBoost"
except Exception as e:
    print(f"xgboost unavailable ({e}) — falling back to sklearn GradientBoostingRegressor")
    def make_gbm():
        return GradientBoostingRegressor(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=RANDOM_STATE)
    GBM_NAME = "GradientBoosting(sklearn)"

print("Gradient boosting model in use:", GBM_NAME)

RIDGE_ALPHA_GRID = (1.0, 3.0, 10.0, 30.0, 100.0, 300.0)   # shifted higher than earlier
                                                            # notebooks — TF-IDF features
                                                            # are high-dimensional and thin,
                                                            # need heavier shrinkage

def tune_ridge_alpha(X_train, y_train, alphas=RIDGE_ALPHA_GRID, min_months_for_cv=24):
    n = len(y_train)
    if n < min_months_for_cv:
        return alphas[len(alphas) // 2], {}
    n_splits = min(5, max(2, n // 12))
    tscv = TimeSeriesSplit(n_splits=n_splits)
    scores = {}
    for alpha in alphas:
        fold_rmses = []
        for tr_idx, val_idx in tscv.split(X_train):
            if len(tr_idx) < 3 or len(val_idx) < 1:
                continue
            m = Ridge(alpha=alpha).fit(X_train[tr_idx], y_train[tr_idx])
            pred = m.predict(X_train[val_idx])
            fold_rmses.append(mean_squared_error(y_train[val_idx], pred) ** 0.5)
        scores[alpha] = np.mean(fold_rmses) if fold_rmses else np.inf
    best_alpha = min(scores, key=scores.get)
    return best_alpha, scores

ELASTICNET_ALPHA_GRID = (0.01, 0.03, 0.1, 0.3, 1.0, 3.0)
ELASTICNET_L1_RATIO_GRID = (0.1, 0.5, 0.9)   # 0.1 = mostly Ridge-like, 0.9 = mostly Lasso-like

def tune_elasticnet(X_train, y_train, alphas=ELASTICNET_ALPHA_GRID, l1_ratios=ELASTICNET_L1_RATIO_GRID,
                     min_months_for_cv=24):
    """Same train-only TimeSeriesSplit discipline as tune_ridge_alpha — this is the model
    Bortoli, Combes & Renault (2018) actually used for word-frequency regressors, so it's
    a direct replication of their method rather than a Ridge approximation of it."""
    n = len(y_train)
    if n < min_months_for_cv:
        return alphas[len(alphas) // 2], l1_ratios[len(l1_ratios) // 2]
    n_splits = min(5, max(2, n // 12))
    tscv = TimeSeriesSplit(n_splits=n_splits)
    best_params, best_rmse = (alphas[0], l1_ratios[0]), np.inf
    for alpha in alphas:
        for l1_ratio in l1_ratios:
            fold_rmses = []
            for tr_idx, val_idx in tscv.split(X_train):
                if len(tr_idx) < 3 or len(val_idx) < 1:
                    continue
                m = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=5000).fit(X_train[tr_idx], y_train[tr_idx])
                pred = m.predict(X_train[val_idx])
                fold_rmses.append(mean_squared_error(y_train[val_idx], pred) ** 0.5)
            mean_rmse = np.mean(fold_rmses) if fold_rmses else np.inf
            if mean_rmse < best_rmse:
                best_rmse, best_params = mean_rmse, (alpha, l1_ratio)
    return best_params

def evaluate(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    direction_match = np.sign(y_true) == np.sign(y_pred)
    return {
        "rmse": mean_squared_error(y_true, y_pred) ** 0.5,
        "mae": mean_absolute_error(y_true, y_pred),
        "r2": r2_score(y_true, y_pred) if len(y_true) > 1 else float("nan"),
        "directional_accuracy": direction_match.mean(),
    }

def naive_persistence_forecast(y_train, n_test):
    return np.full(n_test, y_train[-1])

def select_arima_order(y_train, candidate_orders=((1, 0, 0), (1, 0, 1))):
    best_order, best_aic = (1, 0, 0), float("inf")
    for order in candidate_orders:
        try:
            fit = ARIMA(y_train, order=order).fit()
            if fit.aic < best_aic:
                best_aic, best_order = fit.aic, order
        except Exception:
            continue
    return best_order

def fit_arima_baseline(y_train, n_test, order=None):
    if order is None:
        order = select_arima_order(y_train)
    try:
        model = ARIMA(y_train, order=order).fit()
        return np.asarray(model.forecast(steps=n_test)), order
    except Exception as e:
        print(f"ARIMA failed ({e}) — falling back to naive baseline")
        return naive_persistence_forecast(y_train, n_test), order

Gradient boosting model in use: XGBoost


## Main loop — per outlet, per lag: TF-IDF (fit on TRAIN text only) + Ridge / RF / GBM + baselines

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression

def select_features(X_train, y_train, X_test, k=N_SELECTED_FEATURES):
    """Scores every candidate term by how well it actually correlates with GDP growth,
    using TRAINING rows only, and keeps just the top k. Fit only ever touches training
    data — scoring test-period terms against test-period growth would leak the answer."""
    k = min(k, X_train.shape[1])
    selector = SelectKBest(score_func=f_regression, k=k)
    X_train_sel = selector.fit_transform(X_train, y_train)
    X_test_sel = selector.transform(X_test)
    return X_train_sel, X_test_sel, selector

def build_lagged_table(outlet, lag_months):
    """Row m = text from month m, paired with GDP growth at month (m + lag_months).
    lag_months=0 means same-month (nowcasting-style); >0 means genuine forward prediction.
    Also attaches the UK EPU value known as of month m (asof = last available value at or
    before m, so no lookahead), aligned with the same lag logic as the text."""
    corpus = monthly_corpora[outlet]
    rows = []
    for m, doc in corpus.items():
        target_date = m + pd.DateOffset(months=lag_months)
        if target_date in econ.index and pd.notna(econ.loc[target_date, "growth"]):
            epu_val = epu.asof(m) if not epu.empty else np.nan
            rows.append({"text_month": m, "target_date": target_date, "doc": doc,
                         "growth": econ.loc[target_date, "growth"], "epu": epu_val})
    return pd.DataFrame(rows).sort_values("text_month").reset_index(drop=True)

results = []

for outlet in OUTLETS_PLUS_POOLED:  # includes the pooled (all-outlet) corpus
    for lag in LAGS_MONTHS:
        aligned = build_lagged_table(outlet, lag)
        n = len(aligned)
        print(f"\n=== {outlet} / lag={lag}mo ({n} aligned months) ===")
        if n < 20:
            print("Too few months — skipping.")
            continue

        split_idx = int(n * (1 - TEST_FRACTION))
        train_docs, test_docs = aligned["doc"].iloc[:split_idx], aligned["doc"].iloc[split_idx:]
        y = aligned["growth"].values

        # vectorizer is fit ONLY on training-period text — see the leak-prevention
        # note at the top of this notebook
        vectorizer = TfidfVectorizer(max_features=TFIDF_CANDIDATE_FEATURES, min_df=TFIDF_MIN_DF,
                                      stop_words="english", lowercase=True)
        X_train_wide = vectorizer.fit_transform(train_docs).toarray()
        X_test_wide = vectorizer.transform(test_docs).toarray()
        y_train, y_test = y[:split_idx], y[split_idx:]
        print(f"Candidate vocabulary (train-only): {len(vectorizer.vocabulary_)}")

        X_train, X_test, selector = select_features(X_train_wide, y_train, X_test_wide)
        selected_terms = np.array(vectorizer.get_feature_names_out())[selector.get_support()]
        print(f"Selected {X_train.shape[1]} features (top by correlation with growth): {list(selected_terms)}")

        alpha, _ = tune_ridge_alpha(X_train, y_train)
        ridge_pred = Ridge(alpha=alpha).fit(X_train, y_train).predict(X_test)
        results.append({"outlet": outlet, "lag_months": lag, "model": "Ridge (term-count)",
                         **evaluate(y_test, ridge_pred), "n_months": n, "tuned_param": f"alpha={alpha}"})

        # ElasticNet — the actual model Bortoli, Combes & Renault (2018) used for
        # word-frequency regressors, not a Ridge approximation of it
        en_alpha, en_l1 = tune_elasticnet(X_train, y_train)
        en_pred = ElasticNet(alpha=en_alpha, l1_ratio=en_l1, max_iter=5000).fit(X_train, y_train).predict(X_test)
        results.append({"outlet": outlet, "lag_months": lag, "model": "ElasticNet (term-count)",
                         **evaluate(y_test, en_pred), "n_months": n, "tuned_param": f"alpha={en_alpha},l1_ratio={en_l1}"})

        rf = RandomForestRegressor(n_estimators=200, max_depth=5, random_state=RANDOM_STATE).fit(X_train, y_train)
        rf_pred = rf.predict(X_test)
        results.append({"outlet": outlet, "lag_months": lag, "model": "RandomForest (term-count)",
                         **evaluate(y_test, rf_pred), "n_months": n, "tuned_param": "n_estimators=200,max_depth=5"})

        gbm = make_gbm().fit(X_train, y_train)
        gbm_pred = gbm.predict(X_test)
        results.append({"outlet": outlet, "lag_months": lag, "model": f"{GBM_NAME} (term-count)",
                         **evaluate(y_test, gbm_pred), "n_months": n, "tuned_param": "n_estimators=200,max_depth=3"})

        # EPU-augmented variants — same selected term features + the UK EPU index as one
        # extra column. EPU's train-period mean fills any gaps; filled using training
        # data only, same leak-prevention discipline as everything else here.
        epu_train = aligned["epu"].values[:split_idx].astype(float)
        epu_test = aligned["epu"].values[split_idx:].astype(float)
        epu_fill = np.nanmean(epu_train) if not np.all(np.isnan(epu_train)) else 0.0
        epu_train = np.nan_to_num(epu_train, nan=epu_fill).reshape(-1, 1)
        epu_test = np.nan_to_num(epu_test, nan=epu_fill).reshape(-1, 1)
        X_train_epu_raw = np.hstack([X_train, epu_train])
        X_test_epu_raw = np.hstack([X_test, epu_test])

        # TF-IDF features are ~0-1, EPU is in the hundreds — without scaling, Ridge/
        # ElasticNet's penalty (which treats all coefficient sizes equally) gets
        # distorted by the scale mismatch. StandardScaler is fit on TRAINING rows only,
        # same leak-prevention discipline as everything else in this notebook.
        epu_scaler = StandardScaler().fit(X_train_epu_raw)
        X_train_epu = epu_scaler.transform(X_train_epu_raw)
        X_test_epu = epu_scaler.transform(X_test_epu_raw)

        alpha_e, _ = tune_ridge_alpha(X_train_epu, y_train)
        ridge_epu_pred = Ridge(alpha=alpha_e).fit(X_train_epu, y_train).predict(X_test_epu)
        results.append({"outlet": outlet, "lag_months": lag, "model": "Ridge (term-count+EPU)",
                         **evaluate(y_test, ridge_epu_pred), "n_months": n, "tuned_param": f"alpha={alpha_e}"})

        en_alpha2, en_l12 = tune_elasticnet(X_train_epu, y_train)
        en_epu_pred = ElasticNet(alpha=en_alpha2, l1_ratio=en_l12, max_iter=5000).fit(X_train_epu, y_train).predict(X_test_epu)
        results.append({"outlet": outlet, "lag_months": lag, "model": "ElasticNet (term-count+EPU)",
                         **evaluate(y_test, en_epu_pred), "n_months": n, "tuned_param": f"alpha={en_alpha2},l1_ratio={en_l12}"})

        naive_pred = naive_persistence_forecast(y_train, len(y_test))
        results.append({"outlet": outlet, "lag_months": lag, "model": "Naive (no-change)",
                         **evaluate(y_test, naive_pred), "n_months": n, "tuned_param": ""})

        arima_pred, arima_order = fit_arima_baseline(y_train, len(y_test))
        results.append({"outlet": outlet, "lag_months": lag, "model": "ARIMA(auto)",
                         **evaluate(y_test, arima_pred), "n_months": n, "tuned_param": f"order={arima_order}"})

results_df = pd.DataFrame(results)
print("\n\n=== FINAL RESULTS (term-count features, all lags) ===")
print(results_df)
results_df.to_csv(f"{DATA_DIR}/model_results_termcount_lags_TEST.csv", index=False)
print(f"\nSaved to {DATA_DIR}/model_results_termcount_lags_TEST.csv")


=== bbc / lag=0mo (109 aligned months) ===
Candidate vocabulary (train-only): 300
Selected 30 features (top by correlation with growth): ['20', '2021', 'ahead', 'boris', 'coronavirus', 'của', 'deal', 'december', 'del', 'el', 'en', 'la', 'latest', 'local', 'los', 'making', 'minister', 'para', 'people', 'por', 'published', 'que', 'said', 'se', 'street', 'told', 'trade', 'wedi', 'wednesday', 'yn']

=== bbc / lag=1mo (108 aligned months) ===
Candidate vocabulary (train-only): 300
Selected 30 features (top by correlation with growth): ['2020', 'average', 'bank', 'change', 'china', 'comes', 'coronavirus', 'các', 'của', 'deal', 'february', 'group', 'including', 'market', 'measures', 'ministers', 'na', 'nhs', 'place', 'saying', 'says', 'scheme', 'street', 'trade', 'virus', 'và', 'wa', 'welsh', 'world', 'ya']

=== bbc / lag=2mo (107 aligned months) ===
Candidate vocabulary (train-only): 300
Selected 30 features (top by correlation with growth): ['000', '19', 'cases', 'china', 'city', 'coronavi

## Recession-specific test — both windows, per outlet, per lag



In [11]:
recession_results = []

for outlet in OUTLETS_PLUS_POOLED:  # includes the pooled (all-outlet) corpus
    for lag in LAGS_MONTHS:
        aligned = build_lagged_table(outlet, lag)
        if len(aligned) < 20:
            continue

        for window in RECESSION_WINDOWS:
            rec_start, rec_end, label = window["start"], window["end"], window["name"]
            leadup_start = rec_start - pd.DateOffset(months=LEADUP_MONTHS)
            calm_end = rec_end + pd.DateOffset(months=CALM_COMPARISON_MONTHS)

            train = aligned[aligned["target_date"] < rec_start]
            test_leadup = aligned[(aligned["target_date"] >= leadup_start) & (aligned["target_date"] < rec_start)]
            test_recession = aligned[(aligned["target_date"] >= rec_start) & (aligned["target_date"] <= rec_end)]
            test_calm = aligned[(aligned["target_date"] > rec_end) & (aligned["target_date"] <= calm_end)]

            tag = f"{outlet}/lag={lag}mo [{label}]"
            if len(train) < 24 or len(test_recession) < 2:
                print(f"{tag}: {len(train)} training months — not enough data, skipping")
                continue

            vectorizer = TfidfVectorizer(max_features=TFIDF_CANDIDATE_FEATURES, min_df=TFIDF_MIN_DF,
                                          stop_words="english", lowercase=True)
            X_train_wide = vectorizer.fit_transform(train["doc"]).toarray()
            y_train_r = train["growth"].values

            # same train-only selection discipline as the main loop — score candidate
            # terms against training growth only, then reuse that fitted selector
            # (never refit) on every test period below
            k = min(N_SELECTED_FEATURES, X_train_wide.shape[1])
            selector = SelectKBest(score_func=f_regression, k=k).fit(X_train_wide, y_train_r)
            X_train = selector.transform(X_train_wide)

            epu_train_r = train["epu"].values.astype(float)
            epu_fill_r = np.nanmean(epu_train_r) if not np.all(np.isnan(epu_train_r)) else 0.0
            epu_train_r_filled = np.nan_to_num(epu_train_r, nan=epu_fill_r).reshape(-1, 1)
            X_train_epu_raw = np.hstack([X_train, epu_train_r_filled])

            # same scale-mismatch fix as the main loop — fit on training rows only
            epu_scaler_r = StandardScaler().fit(X_train_epu_raw)
            X_train_epu = epu_scaler_r.transform(X_train_epu_raw)

            alpha_r, _ = tune_ridge_alpha(X_train, y_train_r)
            ridge_model = Ridge(alpha=alpha_r).fit(X_train, y_train_r)

            en_alpha_r, en_l1_r = tune_elasticnet(X_train, y_train_r)
            en_model = ElasticNet(alpha=en_alpha_r, l1_ratio=en_l1_r, max_iter=5000).fit(X_train, y_train_r)

            alpha_re, _ = tune_ridge_alpha(X_train_epu, y_train_r)
            ridge_epu_model = Ridge(alpha=alpha_re).fit(X_train_epu, y_train_r)

            print(f"{tag}: {len(train)} training months, {k} selected features, "
                  f"Ridge alpha={alpha_r}, ElasticNet alpha={en_alpha_r}/l1={en_l1_r}, Ridge+EPU alpha={alpha_re}")

            models_to_test = [
                ("Ridge (term-count)", ridge_model, False),
                ("ElasticNet (term-count)", en_model, False),
                ("Ridge (term-count+EPU)", ridge_epu_model, True),
            ]

            for period_name, test_set in [
                ("lead-up (pre-recession)", test_leadup),
                ("recession", test_recession),
                ("calm (post-recession)", test_calm),
            ]:
                if len(test_set) < 2:
                    continue
                X_test_wide = vectorizer.transform(test_set["doc"]).toarray()
                X_test = selector.transform(X_test_wide)

                epu_test_r = test_set["epu"].values.astype(float)
                epu_test_r_filled = np.nan_to_num(epu_test_r, nan=epu_fill_r).reshape(-1, 1)
                X_test_epu_raw = np.hstack([X_test, epu_test_r_filled])
                X_test_epu = epu_scaler_r.transform(X_test_epu_raw)

                for model_name, fitted_model, uses_epu in models_to_test:
                    X_eval = X_test_epu if uses_epu else X_test
                    pred = fitted_model.predict(X_eval)
                    recession_results.append({
                        "outlet": outlet, "lag_months": lag, "model": model_name,
                        "recession": label, "period": period_name,
                        **evaluate(test_set["growth"].values, pred),
                        "n_months": len(test_set), "tuned_param": "",
                    })

recession_df = pd.DataFrame(recession_results)
print("\n\n=== LEAD-UP vs RECESSION vs CALM COMPARISON (term-count features, all lags) ===")
print(recession_df)
recession_df.to_csv(f"{DATA_DIR}/recession_specific_results_termcount_lags_TEST.csv", index=False)
print(f"\nSaved to {DATA_DIR}/recession_specific_results_termcount_lags_TEST.csv")

bbc/lag=0mo [2020 COVID recession]: 44 training months, 30 selected features, Ridge alpha=1.0, ElasticNet alpha=0.03/l1=0.1, Ridge+EPU alpha=100.0
bbc/lag=0mo [2023 H2 recession]: 74 training months, 30 selected features, Ridge alpha=300.0, ElasticNet alpha=0.3/l1=0.9, Ridge+EPU alpha=300.0
bbc/lag=1mo [2020 COVID recession]: 44 training months, 30 selected features, Ridge alpha=1.0, ElasticNet alpha=0.01/l1=0.1, Ridge+EPU alpha=30.0
bbc/lag=1mo [2023 H2 recession]: 73 training months, 30 selected features, Ridge alpha=300.0, ElasticNet alpha=0.3/l1=0.5, Ridge+EPU alpha=300.0
bbc/lag=2mo [2020 COVID recession]: 44 training months, 30 selected features, Ridge alpha=1.0, ElasticNet alpha=0.1/l1=0.5, Ridge+EPU alpha=10.0
bbc/lag=2mo [2023 H2 recession]: 72 training months, 30 selected features, Ridge alpha=300.0, ElasticNet alpha=0.1/l1=0.9, Ridge+EPU alpha=300.0
bbc/lag=3mo [2020 COVID recession]: 44 training months, 30 selected features, Ridge alpha=1.0, ElasticNet alpha=0.01/l1=0.1, Ri

## Clear, separated results tables

In [12]:
print("=== MAIN RESULTS, by outlet / lag ===\n")
for (outlet, lag), sub in results_df.groupby(["outlet", "lag_months"]):
    print(f"--- {outlet} / lag={lag}mo ---")
    print(sub[["model", "rmse", "mae", "r2", "directional_accuracy", "n_months", "tuned_param"]].to_string(index=False))
    print()

if len(recession_df):
    print("\n=== RECESSION-SPECIFIC RESULTS, by outlet / lag / recession ===\n")
    for (outlet, lag, rec), sub in recession_df.groupby(["outlet", "lag_months", "recession"]):
        print(f"--- {outlet} / lag={lag}mo / {rec} ---")
        print(sub[["model", "period", "rmse", "mae", "r2", "directional_accuracy", "n_months"]].to_string(index=False))
        print()
else:
    print("\nNo outlet/lag combination had 24+ pre-recession training months for the recession-specific test.")

print("\n=== Quick scan: best directional accuracy per outlet (main loop, across all lags/models) ===\n")
if len(results_df):
    best = results_df.loc[results_df.groupby("outlet")["directional_accuracy"].idxmax()]
    print(best[["outlet", "lag_months", "model", "directional_accuracy", "r2", "n_months"]].to_string(index=False))

=== MAIN RESULTS, by outlet / lag ===

--- bbc / lag=0mo ---
                      model     rmse      mae          r2  directional_accuracy  n_months                  tuned_param
         Ridge (term-count) 0.321739 0.260994   -1.759520              0.272727       109                  alpha=300.0
    ElasticNet (term-count) 0.316695 0.255323   -1.673663              0.272727       109       alpha=0.3,l1_ratio=0.9
  RandomForest (term-count) 1.307629 0.839987  -44.582149              0.363636       109 n_estimators=200,max_depth=5
       XGBoost (term-count) 3.587130 1.094769 -342.020366              0.363636       109 n_estimators=200,max_depth=3
     Ridge (term-count+EPU) 0.976026 0.754754  -24.395006              0.227273       109                  alpha=300.0
ElasticNet (term-count+EPU) 0.316695 0.255323   -1.673663              0.272727       109       alpha=3.0,l1_ratio=0.9
          Naive (no-change) 0.221370 0.169496   -0.306355              0.181818       109                 